In [1]:
!pip install requests beautifulsoup4 pandas lxml

In [3]:
import re
import time
import os
import datetime as dt
import pandas as pd
import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin
from urllib.robotparser import RobotFileParser

HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
                  "AppleWebKit/537.36 (KHTML, like Gecko) "
                  "Chrome/120.0 Safari/537.36"
}

FIELDNAMES = [
    "programme_id", "university", "programme_name", "year", "country",
    "region", "school_type", "college_type", "ranking_tier",
    "statistics_core", "programming_core", "machine_learning_core",
    "data_visualization_core", "business_analytics_core",
    "core_category_count", "prerequisites", "GRE_GMAT_policy",
    "tuition_amount", "currency", "duration_months", "outcomes_reported",
    "source_type", "source_url", "access_date", "english_requirement",
    "work_experience_required", "application_fee", "minimum_gpa",
    "non_business_background_accepted", "study_mode", "application_deadline",
]

all_rows = []

In [4]:
def check_robots(url, agent=HEADERS["User-Agent"], show=10):
    robots_url = urljoin(url, "/robots.txt")
    try:
        r = requests.get(robots_url, headers=HEADERS, timeout=10)
    except requests.RequestException as e:
        print(f"could not reach {robots_url}: {type(e).__name__}")
        return None

    if r.status_code == 404:
        print("no robots.txt published --- no rules to follow")
        return True
    if r.status_code >= 400:
        print(f"robots.txt returned {r.status_code} --- cannot tell")
        return None

    keep = ("user-agent", "disallow", "allow", "crawl-delay")
    rules = [ln.strip() for ln in r.text.splitlines()
             if ln.strip().lower().startswith(keep)]

    print(f"{robots_url}  ({len(rules)} rules)")
    for line in rules[:show]:
        print("   ", line)
    if len(rules) > show:
        print(f"    ... {len(rules) - show} more")

    rp = RobotFileParser()
    rp.parse(r.text.splitlines())
    verdict = rp.can_fetch(agent, url)
    print(f"\nPython's parser says: {verdict}")
    return verdict

In [9]:
MONEY_RE = re.compile(
    r"(?:US|S|A|C|HK)?\$\s?[\d,]+(?:\.\d{1,2})?"
    r"|\d{1,3}(?:,\d{3})+(?:\.\d{1,2})?\s?(?:USD|SGD|GBP|EUR|AUD|CAD|HKD|RMB|CNY)"
)

CURRENCY_MAP = {
    "$": "USD", "US$": "USD", "S$": "SGD", "SGD": "SGD",
    "£": "GBP", "GBP": "GBP", "€": "EUR", "EUR": "EUR",
    "A$": "AUD", "AUD": "AUD", "C$": "CAD", "CAD": "CAD",
    "HK$": "HKD", "HKD": "HKD", "¥": "CNY", "RMB": "CNY", "CNY": "CNY",
}

def detect_currency(s):
    if not s:
        return ""
    for k, v in CURRENCY_MAP.items():
        if k in s:
            return v
    return ""

def to_num(x):
    x = re.sub(r"[^\d.]", "", str(x))
    try:
        return float(x)
    except ValueError:
        return 0.0

DURATION_RE = re.compile(
    r"(?i)(\d+(?:\.\d+)?)\s*(?:-|–|to)?\s*(\d+(?:\.\d+)?)?\s*"
    r"(months?|years?|semesters?|terms?|trimesters?)"
)

DURATION_UNIT_TO_MONTHS = {
    "month": 1, "months": 1,
    "year": 12, "years": 12,
    "semester": 6, "semesters": 6,
    "term": 3, "terms": 3,
    "trimester": 4, "trimesters": 4,
}

DEADLINE_RE = re.compile(
    r"(?i)(?:"
    r"Round\s*\d+[^.;\n]{0,60}|"
    r"Term\s*\d+[^.;\n]{0,60}|"
    r"Intake[^.;\n]{0,60}|"
    r"Application\s+(?:deadline|period|window)[^.;\n]{0,60}|"
    r"Deadline[^.;\n]{0,60}|"
    r"Closing\s+date[^.;\n]{0,60}"
    r")"
)



IELTS_RE = re.compile(r"(?i)IELTS[^.\n]{0,80}|TOEFL[^.\n]{0,80}|"
                      r"English\s+(?:language\s+)?(?:requirement|proficiency)[^.\n]{0,80}")

WORK_EXP_RE = re.compile(r"(?i)(?:work\s+experience|professional\s+experience|"
                         r"years?\s+of\s+experience)[^.\n]{0,80}")

APP_FEE_RE = re.compile(r"(?i)(?:application|admission)\s+fee[^.\n]{0,60}")

STUDY_MODE_RE = re.compile(r"(?i)(full[- ]time|part[- ]time|on[- ]campus|"
                           r"online|blended|hybrid|face[- ]to[- ]face)[^.\n]{0,40}")

In [7]:
KW = {
    "statistics": ["statistics", "statistical", "probability", "regression",
                   " econometrics", "quantitative method"],
    "programming": ["programming", "python", "r programming", "sql",
                    "java", "coding", "software", "data structures"],
    "machine_learning": ["machine learning", "deep learning", "ai",
                         "artificial intelligence", "neural network",
                         "predictive model", "data mining"],
    "data_visualization": ["data visualization", "visualisation",
                           "tableau", "power bi", "dashboard",
                           "visual analytics", "ggplot"],
    "business_analytics": ["business analytics", "business intelligence",
                           "analytics strategy", "decision making",
                           "business decision", "marketing analytics",
                           "financial analytics"],
    "prerequisites": ["prerequisite", "admission requirement",
                      "entry requirement", "admission criteria",
                      "eligibility", "minimum requirement",
                      "academic requirement", "background required",
                      "prior knowledge", "quantitative background",
                      "programming background"],
    "test_policy": ["gmat", "gre", "test score", "standardized test",
                    "waiver", "waived", "not required", "optional",
                    "test optional", "no gmat", "no gre"],
    "tuition": ["tuition", "tuition fee", "programme fee", "program fee",
                "total fee", "course fee", "fees", "cost",
                "indicative fee", "first year fee", "per year",
                "per credit"],
    "outcomes": ["employment report", "career report", "career outcome",
                 "employment outcome", "placement", "salary",
                 "average salary", "median salary", "graduate outcome",
                 "job outcome", "career prospect", "employment rate"],
    "english": ["ielts", "toefl", "english language", "english proficiency"],
    "work_exp": ["work experience", "professional experience",
                 "years of experience", "internship"],
    "app_fee": ["application fee", "admission fee"],
    "gpa": ["minimum gpa", "gpa", "grade point average"],
    "non_business": ["non-business", "non business", "any discipline",
                     "any background", "all backgrounds",
                     "no business background required"],
    "study_mode": ["full-time", "part-time", "on-campus", "online",
                   "blended", "hybrid", "face-to-face"],
    "deadline": ["deadline", "application period", "round", "intake",
                 "closing date", "term"],
}

def find_contexts(text, keywords, window=250, max_hits=3):
    hits = []
    low = text.lower()
    for kw in keywords:
        start = 0
        while True:
            i = low.find(kw.lower(), start)
            if i == -1:
                break
            left = max(0, i - window)
            right = min(len(text), i + len(kw) + window)
            hits.append(text[left:right].strip())
            start = i + len(kw)
            if len(hits) >= max_hits * 3:
                break
    seen, out = set(), []
    for h in hits:
        key = h[:120]
        if key not in seen:
            seen.add(key)
            out.append(h)
        if len(out) >= max_hits:
            break
    return " || ".join(out)

In [8]:
def parse_programme(url, meta=None):
    """
    meta: dict，包含 programme_id, university, year, country, region,
          school_type, college_type, ranking_tier, source_type
    """
    meta = meta or {}

    r = requests.get(url, headers=HEADERS, timeout=25)
    r.encoding = r.apparent_encoding
    soup = BeautifulSoup(r.text, "html.parser")
    for tag in soup(["script", "style", "noscript"]):
        tag.decompose()
    text = soup.get_text(" ", strip=True)

    # ---------- 标题 ----------
    h1 = soup.find("h1")
    programme_name = h1.get_text(strip=True) if h1 else ""
    if not programme_name:
        title = soup.find("title")
        programme_name = title.get_text(strip=True) if title else ""

    # ---------- 学费 ----------
    money_hits = MONEY_RE.findall(text)
    tuition_ctx = find_contexts(text, KW["tuition"], window=300, max_hits=3)
    ctx_money = MONEY_RE.findall(tuition_ctx) if tuition_ctx else []
    tuition_candidates = ctx_money or money_hits
    tuition_amount = ""
    if tuition_candidates:
        # 取上下文里最大的金额，通常最接近学费
        try:
            tuition_amount = max(tuition_candidates, key=to_num)
        except ValueError:
            tuition_amount = tuition_candidates[0]
    currency = detect_currency(tuition_amount) or detect_currency(tuition_ctx)

    # ---------- 时长 ----------
    duration_months = ""
    for m in DURATION_RE.finditer(text):
        unit = m.group(3).lower()
        if unit in DURATION_UNIT_TO_MONTHS:
            val = float(m.group(1))
            months = int(round(val * DURATION_UNIT_TO_MONTHS[unit]))
            duration_months = months
            break

    # ---------- 课程类别 ----------
    def has_kw(kws):
        low = text.lower()
        return int(any(k.lower() in low for k in kws))

    statistics_core = has_kw(KW["statistics"])
    programming_core = has_kw(KW["programming"])
    machine_learning_core = has_kw(KW["machine_learning"])
    data_visualization_core = has_kw(KW["data_visualization"])
    business_analytics_core = has_kw(KW["business_analytics"])
    core_category_count = sum([
        statistics_core, programming_core, machine_learning_core,
        data_visualization_core, business_analytics_core
    ])

    # ---------- 先修课 ----------
    prereq_ctx = find_contexts(text, KW["prerequisites"], window=300, max_hits=3)

    # ---------- GRE/GMAT ----------
    test_ctx = find_contexts(text, KW["test_policy"], window=200, max_hits=4)
    GRE_GMAT_policy = ""
    low = text.lower()
    if "gmat" in low or "gre" in low:
        if re.search(r"(?i)(not required|waiv|optional|no gmat|no gre)", text):
            GRE_GMAT_policy = "optional/waived"
        elif re.search(r"(?i)(required|mandatory)", text):
            GRE_GMAT_policy = "required"
        else:
            GRE_GMAT_policy = "mentioned"

    # ---------- 就业公开 ----------
    outcomes_ctx = find_contexts(text, KW["outcomes"], window=250, max_hits=3)
    outcomes_reported = 1 if outcomes_ctx else 0

    # ---------- 英语要求 ----------
    english_ctx = find_contexts(text, KW["english"], window=200, max_hits=2)

    # ---------- 工作经验 ----------
    work_ctx = find_contexts(text, KW["work_exp"], window=200, max_hits=2)
    work_experience_required = 1 if re.search(r"(?i)(required|minimum)", work_ctx) else 0

    # ---------- 申请费 ----------
    app_fee_ctx = find_contexts(text, KW["app_fee"], window=150, max_hits=2)
    app_fee_candidates = MONEY_RE.findall(app_fee_ctx)
    application_fee = app_fee_candidates[0] if app_fee_candidates else ""

    # ---------- GPA ----------
    gpa_ctx = find_contexts(text, KW["gpa"], window=200, max_hits=2)
    minimum_gpa = ""
    m = re.search(r"(?i)(?:minimum\s+)?GPA[^.\n]{0,60}", text)
    if m:
        minimum_gpa = m.group(0).strip()

    # ---------- 非商科背景 ----------
    non_biz_ctx = find_contexts(text, KW["non_business"], window=200, max_hits=2)
    non_business_background_accepted = 1 if non_biz_ctx else 0

    # ---------- 学习模式 ----------
    study_mode_ctx = find_contexts(text, KW["study_mode"], window=150, max_hits=3)

    # ---------- 截止日期 ----------
    deadlines = list(dict.fromkeys(DEADLINE_RE.findall(text)))
    application_deadline = " | ".join(deadlines)[:500]

    access_date = dt.date.today().isoformat()

    return {
        "programme_id": meta.get("programme_id", ""),
        "university": meta.get("university", ""),
        "programme_name": programme_name,
        "year": meta.get("year", ""),
        "country": meta.get("country", ""),
        "region": meta.get("region", ""),
        "school_type": meta.get("school_type", ""),
        "college_type": meta.get("college_type", ""),
        "ranking_tier": meta.get("ranking_tier", ""),

        "statistics_core": statistics_core,
        "programming_core": programming_core,
        "machine_learning_core": machine_learning_core,
        "data_visualization_core": data_visualization_core,
        "business_analytics_core": business_analytics_core,
        "core_category_count": core_category_count,

        "prerequisites": prereq_ctx[:800],
        "GRE_GMAT_policy": GRE_GMAT_policy,
        "tuition_amount": tuition_amount,
        "currency": currency,
        "duration_months": duration_months,
        "outcomes_reported": outcomes_reported,

        "source_type": meta.get("source_type", "official programme page"),
        "source_url": url,
        "access_date": access_date,

        "english_requirement": english_ctx[:500],
        "work_experience_required": work_experience_required,
        "application_fee": application_fee,
        "minimum_gpa": minimum_gpa,
        "non_business_background_accepted": non_business_background_accepted,
        "study_mode": study_mode_ctx[:300],
        "application_deadline": application_deadline,

        # 额外保留，方便人工核对
        "_tuition_context": tuition_ctx[:800],
        "_test_context": test_ctx[:800],
        "_outcomes_context": outcomes_ctx[:800],
        "_gpa_context": gpa_ctx[:500],
        "_non_biz_context": non_biz_ctx[:500],
        "_full_text_snippet": text[:3000],
    }

In [10]:
def crawl_one(url, meta=None, save_every=True):
    print("=" * 80)
    print("URL:", url)
    print("=" * 80)

    allowed = check_robots(url)
    if allowed is False:
        print("robots.txt says NO. 跳过。")
        return None
    if allowed is None:
        print("无法判断 robots.txt，先继续，但你要自己确认。")

    try:
        row = parse_programme(url, meta=meta)
    except Exception as e:
        print("爬取出错:", type(e).__name__, e)
        return None

    all_rows.append(row)
    df = pd.DataFrame([row])
    display(df.T)

    if save_every:
        out = pd.DataFrame(all_rows)
        # 保证列顺序
        cols = [c for c in FIELDNAMES if c in out.columns] + \
               [c for c in out.columns if c not in FIELDNAMES]
        out = out[cols]
        out.to_csv("pilot_data_partial.csv", index=False, encoding="utf-8-sig")
        print("已保存 pilot_data_partial.csv")

    return df

In [11]:
url = "https://www.unsw.edu.au/study/postgraduate/graduate-certificate-business-analytics"

meta = {
    "programme_id": "UNSW-GCBA-2026",
    "university": "UNSW Sydney",
    "year": "2026",
    "country": "Australia",
    "region": "Oceania",
    "school_type": "public",
    "college_type": "business",
    "ranking_tier": "top100",
    "source_type": "official programme page",
}

df = crawl_one(url, meta=meta)

URL: https://www.unsw.edu.au/study/postgraduate/graduate-certificate-business-analytics
https://www.unsw.edu.au/robots.txt  (147 rules)
    User-agent: SiteimproveBot
    User-agent: SiteimproveBot-Crawler
    Crawl-delay: 1
    Allow: /misc/*.css$
    Allow: /misc/*.css?
    Allow: /misc/*.js$
    Allow: /misc/*.js?
    Allow: /misc/*.gif
    Allow: /misc/*.jpg
    Allow: /misc/*.jpeg
    ... 137 more

Python's parser says: True


,0
programme_id,UNSW-GCBA-2026
university,UNSW Sydney
programme_name,404 - Page not found
year,2026
country,Australia
region,Oceania
school_type,public
college_type,business
ranking_tier,top100
statistics_core,0


已保存 pilot_data_partial.csv


In [12]:
all_df = pd.DataFrame(all_rows)

# 保证列顺序
cols = [c for c in FIELDNAMES if c in all_df.columns] + \
       [c for c in all_df.columns if c not in FIELDNAMES]
all_df = all_df[cols]

all_df

,programme_id,university,programme_name,year,country,region,school_type,college_type,ranking_tier,statistics_core,...,minimum_gpa,non_business_background_accepted,study_mode,application_deadline,_tuition_context,_test_context,_outcomes_context,_gpa_context,_non_biz_context,_full_text_snippet
0,UNSW-GCBA-2026,UNSW Sydney,404 - Page not found,2026,Australia,Oceania,public,business,top100,0,...,,0,404 - Page not found Skip to Main Content Stud...,,elp me choose Professional development Short c...,404 - Page not found Skip to Main Content Stud...,,,,404 - Page not found Skip to Main Content Stud...


In [ ]:
all_df.to_csv("pilot_data.csv", index=False, encoding="utf-8-sig")
print("已保存 pilot_data.csv")